# 02 — Empirical Volatility Memory

## Objective

This notebook investigates the persistence structure of SPY volatility across intraday and multi-day horizons.

After removing deterministic intraday seasonality, let

$$
Y_t
=
\log RV_t
-
s(\text{time of day})
$$

denote adjusted log realised variance. The object of interest is its covariance kernel,

$$
K(\tau)
=
\operatorname{Cov}(Y_t,Y_{t+\tau}).
$$

A stationary Ornstein–Uhlenbeck process implies a single exponential covariance kernel,

$$
K(\tau)
=
w e^{-\lambda \tau},
$$

where $\lambda$ determines the rate of mean reversion and

$$
T=\frac{1}{\lambda}
$$

is the corresponding characteristic timescale.

Empirical volatility, however, is known to exhibit persistence across multiple horizons. A natural extension is therefore

$$
K(\tau)
=
\sum_{i=1}^{n} w_i e^{-\lambda_i \tau},
\qquad
w_i \geq 0.
$$

More generally, Bernstein's theorem motivates the representation

$$
K(\tau)
=
\int_0^\infty e^{-\lambda \tau}\,\mu(d\lambda),
$$

where the non-negative measure $\mu$ describes the distribution of relaxation timescales underlying volatility memory.

This notebook therefore asks three questions:

1. Can a single exponential relaxation describe the empirical covariance of SPY volatility?
2. Does a mixture of relaxation timescales provide a materially better representation?
3. Which features of the implied relaxation spectrum are actually identifiable from the data?

The aim is not yet to forecast volatility, but to determine the empirical memory structure that should motivate the forecasting model.

## 1. Load processed data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import sys

sys.path.append("..")


from scipy.optimize import curve_fit

file_path = "../data/SPY_10min_rv.csv"

df = pd.read_csv(file_path)
df["interval_start"] = pd.to_datetime(df["interval_start"])
df["interval_end"] = pd.to_datetime(df["interval_end"])

data = df[df["valid"] & df["log_realised_variance"].notna()].copy()

print("Usable observations:", len(data))
print("Start:", data["interval_start"].min())
print("End:", data["interval_end"].max())

data.head()

Usable observations: 193120
Start: 2002-12-30 09:30:00
End: 2026-09-04 15:40:00


,date,interval_start,interval_end,interval_minutes,expected_returns,observed_returns,realised_variance,log_realised_variance,valid
0,2002-12-30,2002-12-30 09:30:00,2002-12-30 09:40:00,10,10,10,0.000002,-13.024000,True
1,2002-12-30,2002-12-30 09:40:00,2002-12-30 09:50:00,10,10,10,0.000004,-12.443297,True
2,2002-12-30,2002-12-30 09:50:00,2002-12-30 10:00:00,10,10,10,0.000003,-12.577964,True
3,2002-12-30,2002-12-30 10:00:00,2002-12-30 10:10:00,10,10,10,0.000004,-12.340862,True
4,2002-12-30,2002-12-30 10:10:00,2002-12-30 10:20:00,10,10,10,0.000007,-11.897283,True


## 2. Inspect log realised variance

The raw 10-minute observations are noisy, but should display persistent volatility regimes over time.

In [2]:
fig = px.histogram(data, x="log_realised_variance", nbins=150, title="Distribution of 10-Minute Log Realised Variance")
fig.show()

In [3]:
sample = data[(data["interval_start"] >= "2020-01-01") & (data["interval_start"] < "2021-01-01")].copy()

fig = px.scatter(sample, x="interval_start", y="log_realised_variance", title="10-Minute Log Realised Variance - 2020")
fig.show()

## 3. Remove deterministic intraday seasonality

SPY volatility has a strong time-of-day pattern. We subtract the mean log-RV for each 10-minute clock-time block before estimating memory.

In [4]:
data["clock_time"] = data["interval_start"].dt.strftime("%H:%M")

intraday_profile = data.groupby("clock_time").agg(
    mean_log_rv=("log_realised_variance", "mean"),
    median_log_rv=("log_realised_variance", "median"),
    observations=("log_realised_variance", "count")
).reset_index()

fig = px.line(intraday_profile, x="clock_time", y=["mean_log_rv", "median_log_rv"], title="Intraday Log-RV Profile")
fig.show()

In [5]:
intraday_mean = data.groupby("clock_time")["log_realised_variance"].transform("mean")
data["adjusted_log_rv"] = data["log_realised_variance"] - intraday_mean
data["adjusted_log_rv"] = data["adjusted_log_rv"] - data["adjusted_log_rv"].mean()

adjusted_profile = data.groupby("clock_time")["adjusted_log_rv"].mean().reset_index()

fig = px.line(adjusted_profile, x="clock_time", y="adjusted_log_rv", title="Mean Adjusted Log-RV by Time of Day")
fig.show()

print("Overall adjusted mean:", data["adjusted_log_rv"].mean())

Overall adjusted mean: -5.651375528829297e-17


## 4. Estimate empirical memory

We retain both covariance and correlation. Positive-lag covariance is particularly useful because realised-variance measurement noise inflates the observed lag-zero variance.

In [6]:
def intraday_memory(data, max_minutes=360):
    results = []
    x = data[["interval_start", "adjusted_log_rv"]].copy()
    x["date"] = x["interval_start"].dt.date
    variance = np.mean(x["adjusted_log_rv"] ** 2)

    for minutes in range(10, max_minutes + 1, 10):
        lag = pd.Timedelta(minutes=minutes)
        future = x.copy()
        future["interval_start"] = future["interval_start"] - lag
        future = future.rename(columns={"adjusted_log_rv": "future_log_rv"})
        pairs = x.merge(future, on=["date", "interval_start"])
        covariance = np.mean(pairs["adjusted_log_rv"] * pairs["future_log_rv"])

        results.append({
            "lag_minutes": minutes,
            "lag_hours": minutes / 60,
            "covariance": covariance,
            "correlation": covariance / variance,
            "pairs": len(pairs)
        })

    return pd.DataFrame(results)


def interday_memory(data, max_days=30):
    results = []
    x = data[["interval_start", "clock_time", "adjusted_log_rv"]].copy()
    variance = np.mean(x["adjusted_log_rv"] ** 2)

    for days in range(1, max_days + 1):
        lag = pd.Timedelta(days=days)
        future = x.copy()
        future["interval_start"] = future["interval_start"] - lag
        future = future.rename(columns={"adjusted_log_rv": "future_log_rv"})
        pairs = x.merge(future, on=["interval_start", "clock_time"])
        covariance = np.mean(pairs["adjusted_log_rv"] * pairs["future_log_rv"])

        results.append({
            "lag_days": days,
            "lag_hours": days * 24,
            "covariance": covariance,
            "correlation": covariance / variance,
            "pairs": len(pairs)
        })

    return pd.DataFrame(results)

intraday_memory_df = intraday_memory(data)
interday_memory_df = interday_memory(data)

### Intraday memory

In [7]:
fig = px.line(intraday_memory_df, x="lag_hours", y="correlation", markers=True, title="Empirical Volatility Memory - Intraday Lags")
fig.show()

fig = px.line(intraday_memory_df, x="lag_hours", y="pairs", markers=True, title="Available Observation Pairs - Intraday Lags")
fig.show()

### Same-time-of-day interday memory

In [8]:
fig = px.line(interday_memory_df, x="lag_days", y="correlation", markers=True, title="Same-Time-of-Day Volatility Memory")
fig.show()

fig = px.line(interday_memory_df, x="lag_days", y="pairs", markers=True, title="Available Pairs by Calendar-Day Lag")
fig.show()

### Memory across timescales

In [9]:
intraday_plot = intraday_memory_df[["lag_hours", "correlation"]].copy()
intraday_plot["source"] = "Intraday"

interday_plot = interday_memory_df[["lag_hours", "correlation"]].copy()
interday_plot["source"] = "Interday"

combined = pd.concat([intraday_plot, interday_plot], ignore_index=True)

fig = px.scatter(combined, x="lag_hours", y="correlation", color="source", log_x=True, title="Volatility Memory Across Timescales")
fig.show()

## 5. Persistence across market closures

This diagnostic compares each clock-time observation with the next available trading-day observation. It motivates, but does not yet estimate, a slower closed-market volatility clock.

In [10]:
transitions = data[["interval_start", "clock_time", "adjusted_log_rv"]].copy()
transitions = transitions.sort_values(["clock_time", "interval_start"])

transitions["next_time"] = transitions.groupby("clock_time")["interval_start"].shift(-1)
transitions["next_log_rv"] = transitions.groupby("clock_time")["adjusted_log_rv"].shift(-1)
transitions["elapsed_hours"] = (transitions["next_time"] - transitions["interval_start"]).dt.total_seconds() / 3600
transitions["product"] = transitions["adjusted_log_rv"] * transitions["next_log_rv"]

transition_memory = transitions.groupby("elapsed_hours").agg(
    covariance=("product", "mean"),
    pairs=("product", "count")
).reset_index()

variance = np.mean(data["adjusted_log_rv"] ** 2)
transition_memory["correlation"] = transition_memory["covariance"] / variance
transition_memory = transition_memory[transition_memory["pairs"] >= 100].copy()

fig = px.scatter(transition_memory, x="elapsed_hours", y="correlation", size="pairs", text="elapsed_hours", title="Volatility Persistence Across Market Closures")
fig.update_traces(texttemplate="%{text:.0f}h", textposition="top center")
fig.update_layout(xaxis_title="Calendar Hours Until Next Trading-Day Observation", yaxis_title="Correlation")
fig.show()

## 6. Single-OU intraday benchmark

Fit a single exponential to intraday covariance only. This is a short-horizon diagnostic and extrapolation experiment, not the fair model comparison used later.

In [11]:
def ou_covariance(t, weight, mean_reversion):
    return weight * np.exp(-mean_reversion * t)

intraday_lags = intraday_memory_df["lag_hours"].to_numpy()
intraday_covariance = intraday_memory_df["covariance"].to_numpy()

parameters, _ = curve_fit(
    ou_covariance, intraday_lags, intraday_covariance,
    p0=[intraday_covariance[0], 0.1], bounds=(0, np.inf)
)

intraday_weight, intraday_mean_reversion = parameters
intraday_memory_df["single_ou_fit"] = ou_covariance(intraday_lags, intraday_weight, intraday_mean_reversion)

print("Weight:", intraday_weight)
print("Mean reversion:", intraday_mean_reversion)
print("Characteristic timescale:", 1 / intraday_mean_reversion, "hours")

fig = px.line(intraday_memory_df, x="lag_hours", y=["covariance", "single_ou_fit"], title="Empirical Intraday Memory vs Single OU")
fig.show()

Weight: 1.2374278202204778
Mean reversion: 0.03930058569109327
Characteristic timescale: 25.444913413253047 hours


In [12]:
long_lags_hours = np.logspace(np.log10(10 / 60), np.log10(30 * 24), 300)
ou_extrapolation = ou_covariance(long_lags_hours, intraday_weight, intraday_mean_reversion)

intraday_points = intraday_memory_df[["lag_hours", "covariance"]].copy()
intraday_points["source"] = "Intraday"
interday_points = interday_memory_df[["lag_hours", "covariance"]].copy()
interday_points["source"] = "Interday"
empirical = pd.concat([intraday_points, interday_points], ignore_index=True)

fig = px.scatter(empirical, x="lag_hours", y="covariance", color="source", log_x=True, title="Intraday-Fitted Single OU Extrapolation")
fig.add_scatter(x=long_lags_hours, y=ou_extrapolation, mode="lines", name="Single OU")
fig.update_layout(xaxis_title="Calendar Lag (Hours)", yaxis_title="Covariance")
fig.show()

## 7. Fair single-OU vs two-OU comparison

For this first multiscale comparison, closed-market time is assigned zero effective duration. An interday lag of `d` calendar days is therefore represented by `6.5 d` effective open-market hours. This is an exploratory clock convention, not the final model.

In [13]:
intraday_fit = intraday_memory_df[["lag_hours", "covariance", "pairs"]].copy()
intraday_fit["effective_hours"] = intraday_fit["lag_hours"]
intraday_fit["source"] = "Intraday"

interday_fit = interday_memory_df[["lag_days", "covariance", "pairs"]].copy()
interday_fit["effective_hours"] = interday_fit["lag_days"] * 6.5
interday_fit["source"] = "Interday"

fit_data = pd.concat([
    intraday_fit[["effective_hours", "covariance", "pairs", "source"]],
    interday_fit[["effective_hours", "covariance", "pairs", "source"]]
], ignore_index=True).sort_values("effective_hours").reset_index(drop=True)

fig = px.scatter(fit_data, x="effective_hours", y="covariance", color="source", log_x=True, title="Empirical Memory in Effective Market Time")
fig.show()

In [14]:
def two_ou_covariance(t, weight_1, mean_reversion_1, weight_2, mean_reversion_2):
    return weight_1 * np.exp(-mean_reversion_1 * t) + weight_2 * np.exp(-mean_reversion_2 * t)

lags = fit_data["effective_hours"].to_numpy()
covariance = fit_data["covariance"].to_numpy()

single_parameters, _ = curve_fit(
    ou_covariance, lags, covariance,
    p0=[covariance[0], 0.01], bounds=(0, np.inf), maxfev=100000
)
single_weight, single_mean_reversion = single_parameters

two_parameters, _ = curve_fit(
    two_ou_covariance, lags, covariance,
    p0=[0.5, 0.1, 0.8, 0.001], bounds=(0, np.inf), maxfev=100000
)
weight_1, mean_reversion_1, weight_2, mean_reversion_2 = two_parameters

print("Single OU timescale:", 1 / single_mean_reversion, "hours")
print("Fast factor timescale:", 1 / mean_reversion_1, "hours")
print("Slow factor timescale:", 1 / mean_reversion_2, "hours")

Single OU timescale: 289.95090986528146 hours
Fast factor timescale: 2.8715953962977467 hours
Slow factor timescale: 400.5824287519671 hours


In [15]:
single_fit = ou_covariance(lags, single_weight, single_mean_reversion)
two_fit = two_ou_covariance(lags, weight_1, mean_reversion_1, weight_2, mean_reversion_2)

single_mse = np.mean((covariance - single_fit) ** 2)
two_mse = np.mean((covariance - two_fit) ** 2)

print("Single OU MSE:", single_mse)
print("Two OU MSE:", two_mse)
print("MSE reduction:", 1 - two_mse / single_mse)

plot_lags = np.logspace(np.log10(lags.min()), np.log10(lags.max()), 500)

fig = px.scatter(fit_data, x="effective_hours", y="covariance", color="source", log_x=True, title="Single OU vs Two-OU Memory Fit")
fig.add_scatter(x=plot_lags, y=ou_covariance(plot_lags, single_weight, single_mean_reversion), mode="lines", name="Single OU")
fig.add_scatter(x=plot_lags, y=two_ou_covariance(plot_lags, weight_1, mean_reversion_1, weight_2, mean_reversion_2), mode="lines", name="Two OU")
fig.update_layout(xaxis_title="Effective Market Time (Hours)", yaxis_title="Covariance")
fig.show()

Single OU MSE: 0.004025827834089678
Two OU MSE: 0.00031111907050185257
MSE reduction: 0.9227192310939439


## 8. Current findings and next step

The empirical memory curve displays both fast intraday decay and substantial persistence over many days. When both models are fitted to the same combined covariance observations, the two-OU model provides a much better fit than the single OU.

The next experiment is the finite Bernstein approximation

\[K(t) = \sum_i w_i e^{-\lambda_i t}, \qquad w_i \ge 0,\]

using a fixed log-spaced grid of mean-reversion rates and non-negative least squares. The aim is to determine whether the data supports a broad spectrum of timescales or whether the two-factor representation already captures the robust structure.

In [16]:
from scipy.optimize import nnls


minimum_timescale = 10 / 60
maximum_timescale = 2000
number_of_timescales = 30

timescales = np.logspace(
    np.log10(minimum_timescale),
    np.log10(maximum_timescale),
    number_of_timescales
)

mean_reversions = 1 / timescales

basis = np.exp(-np.outer(lags, mean_reversions))

weights, residual_norm = nnls(basis, covariance)

bernstein_fit = basis @ weights
bernstein_mse = np.mean((covariance - bernstein_fit) ** 2)

print("Single OU MSE:", single_mse)
print("Two OU MSE:", two_mse)
print("Bernstein MSE:", bernstein_mse)
print("Active factors:", np.sum(weights > 1e-8))

Single OU MSE: 0.004025827834089678
Two OU MSE: 0.00031111907050185257
Bernstein MSE: 0.00021223988160626032
Active factors: 7


In [17]:
plot_basis = np.exp(-np.outer(plot_lags, mean_reversions))
bernstein_plot = plot_basis @ weights

fig = px.scatter(
    fit_data,
    x="effective_hours",
    y="covariance",
    color="source",
    log_x=True,
    title="Empirical Memory vs Bernstein OU Mixture"
)

fig.add_scatter(
    x=plot_lags,
    y=bernstein_plot,
    mode="lines",
    name="Bernstein Mixture"
)

fig.update_layout(
    xaxis_title="Effective Market Time (Hours)",
    yaxis_title="Covariance"
)

fig.show()

In [18]:
spectrum = pd.DataFrame({
    "timescale_hours": timescales,
    "mean_reversion": mean_reversions,
    "weight": weights
})

fig = px.bar(
    spectrum,
    x="timescale_hours",
    y="weight",
    log_x=True,
    title="Estimated Volatility Memory Spectrum"
)

fig.update_layout(
    xaxis_title="Characteristic Timescale (Hours)",
    yaxis_title="Spectral Weight"
)

fig.show()

In [19]:
results = []
spectra = []

for number_of_timescales in [15, 20, 30, 50, 75]:

    timescales = np.logspace(
        np.log10(10 / 60),
        np.log10(2000),
        number_of_timescales
    )

    mean_reversions = 1 / timescales
    basis = np.exp(-np.outer(lags, mean_reversions))

    weights, _ = nnls(basis, covariance)

    fitted = basis @ weights
    mse = np.mean((covariance - fitted) ** 2)

    results.append({
        "grid_points": number_of_timescales,
        "mse": mse,
        "active_factors": np.sum(weights > 1e-6)
    })

    spectrum = pd.DataFrame({
        "timescale_hours": timescales,
        "weight": weights,
        "grid_points": str(number_of_timescales)
    })

    spectra.append(spectrum)

robustness = pd.DataFrame(results)
spectra = pd.concat(spectra, ignore_index=True)

robustness

,grid_points,mse,active_factors
0,15,0.000215,6
1,20,0.000213,7
2,30,0.000212,7
3,50,0.000212,7
4,75,0.000212,5


In [20]:
fig = px.line(
    spectra,
    x="timescale_hours",
    y="weight",
    color="grid_points",
    log_x=True,
    markers=True,
    title="Memory Spectrum Sensitivity to Grid Resolution"
)

fig.update_layout(
    xaxis_title="Characteristic Timescale (Hours)",
    yaxis_title="Spectral Weight"
)

fig.show()

In [21]:
kernel_fits = []

for number_of_timescales in [15, 20, 30, 50, 75]:
    timescales = np.logspace(
        np.log10(10 / 60),
        np.log10(2000),
        number_of_timescales
    )

    mean_reversions = 1 / timescales
    basis = np.exp(-np.outer(lags, mean_reversions))

    weights, _ = nnls(basis, covariance)

    plot_basis = np.exp(-np.outer(plot_lags, mean_reversions))
    fitted_kernel = plot_basis @ weights

    for lag, fitted_value in zip(plot_lags, fitted_kernel):
        kernel_fits.append({
            "effective_hours": lag,
            "covariance": fitted_value,
            "grid_points": str(number_of_timescales)
        })

kernel_fits = pd.DataFrame(kernel_fits)

In [22]:
fig = px.line(
    kernel_fits,
    x="effective_hours",
    y="covariance",
    color="grid_points",
    log_x=True,
    title="Fitted Kernel Sensitivity to Grid Resolution"
)

fig.show()

In [23]:
def timescale_band(timescale):
    if timescale < 1:
        return "< 1 hour"
    elif timescale < 6:
        return "1 - 6 hours"
    elif timescale < 48:
        return "6 - 48 hours"
    elif timescale < 250:
        return "2 - 10 days"
    else:
        return "> 10 days"


spectra["band"] = spectra["timescale_hours"].apply(timescale_band)

band_weights = spectra.groupby(
    ["grid_points", "band"]
)["weight"].sum().reset_index()

band_order = [
    "< 1 hour",
    "1 - 6 hours",
    "6 - 48 hours",
    "2 - 10 days",
    "> 10 days"
]

band_weights["band"] = pd.Categorical(
    band_weights["band"],
    categories=band_order,
    ordered=True
)

band_weights = band_weights.sort_values(
    ["band", "grid_points"]
)

fig = px.bar(
    band_weights,
    x="band",
    y="weight",
    color="grid_points",
    barmode="group",
    title="Spectral Mass by Timescale Band"
)

fig.update_layout(
    xaxis_title="Characteristic Timescale",
    yaxis_title="Total Spectral Weight"
)

fig.show()

In [24]:
band_totals = band_weights.groupby("grid_points")["weight"].transform("sum")

band_weights["weight_share"] = (
    band_weights["weight"] / band_totals
)

fig = px.bar(
    band_weights,
    x="band",
    y="weight_share",
    color="grid_points",
    barmode="group",
    title="Share of Spectral Mass by Timescale Band"
)

fig.show()

## Conclusion

The empirical covariance structure of deseasonalised SPY log realised variance provides strong evidence that volatility memory cannot be adequately represented by a single relaxation timescale.

A single Ornstein–Uhlenbeck process implies

$$
K(\tau)
=
w e^{-\lambda \tau},
$$

but this specification cannot simultaneously reproduce the rapid covariance decay observed at short horizons and the substantial persistence remaining over multi-day horizons.

Introducing a second relaxation component,

$$
K(\tau)
=
w_f e^{-\lambda_f \tau}
+
w_s e^{-\lambda_s \tau},
$$

produces a substantial improvement in fit, supporting a decomposition of volatility dynamics into relatively fast and persistent components.

The more general Bernstein representation,

$$
K(\tau)
=
\int_0^\infty e^{-\lambda \tau}\,\mu(d\lambda),
$$

provides an even more flexible description of the empirical covariance. However, sensitivity analysis reveals an important identification result: individual spectral weights vary materially as the discretisation of the relaxation-rate grid changes, while the reconstructed covariance kernel remains highly stable.

The inverse problem

$$
K(\tau)
\longrightarrow
\mu(d\lambda)
$$

is therefore ill-posed at fine resolution. The data robustly identifies the existence of multi-timescale volatility memory and the shape of the resulting memory kernel, but not a unique fine-grained relaxation spectrum.

This motivates a parsimonious finite-dimensional approximation rather than attempting to recover every component of the mixing measure.

The next notebook therefore calibrates a two-factor Multi-OU model using training data only, translating the empirical memory structure identified here into a state-space model suitable for filtering and forecasting.